# Gold - Modelo estrella MongoDB para Power BI

Este notebook lee `data/silver/trip_data_normalized/`, construye un modelo estrella agregado y lo carga en MongoDB para consumirlo desde Power BI.

Las dimensiones de catalogo se basan en los diccionarios oficiales de TLC ubicados en `docs/diccionarios/`. La tabla de hechos se agrega antes de cargar a MongoDB para evitar llevar millones de viajes crudos a Power BI.

## 1. Configuracion del entorno

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import os
import re
import subprocess
import urllib.request


def find_project_root(start=None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        normalized = candidate / "data" / "silver" / "trip_data_normalized"
        dictionaries = candidate / "docs" / "diccionarios"
        if normalized.exists() and dictionaries.exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz con data/silver/trip_data_normalized y docs/diccionarios.")


PROJECT_ROOT = find_project_root()
NORMALIZED_BASE_DIR = PROJECT_ROOT / "data" / "silver" / "trip_data_normalized"
LOOKUP_DIR = PROJECT_ROOT / "data" / "lookup" / "taxi_zone_lookup"
LOG_DIR = PROJECT_ROOT / "data" / "logs"
AUDIT_LOG_PATH = LOG_DIR / "gold_mongodb_powerbi_audit.jsonl"
LOG_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_HADOOP_HOME = Path("C:/hadoop")
LOCAL_HADOOP_HOME = (
    SYSTEM_HADOOP_HOME
    if SYSTEM_HADOOP_HOME.exists() or os.access(SYSTEM_HADOOP_HOME.parent, os.W_OK)
    else Path.home() / ".hadoop"
).resolve()
HADOOP_HOME_JAVA = LOCAL_HADOOP_HOME.as_posix()
LOCAL_HADOOP_BIN = LOCAL_HADOOP_HOME / "bin"
LOCAL_HADOOP_BIN.mkdir(parents=True, exist_ok=True)

WINUTILS_BASE_URL = "https://raw.githubusercontent.com/steveloughran/winutils/master/hadoop-3.0.0/bin/"
for hadoop_file in ["winutils.exe", "hadoop.dll"]:
    target = LOCAL_HADOOP_BIN / hadoop_file
    if not target.exists():
        print(f"Descargando {hadoop_file} para PySpark en Windows...")
        urllib.request.urlretrieve(WINUTILS_BASE_URL + hadoop_file, target)

os.environ["HADOOP_HOME"] = HADOOP_HOME_JAVA
os.environ["hadoop.home.dir"] = HADOOP_HOME_JAVA
os.environ["PATH"] = str(LOCAL_HADOOP_BIN) + os.pathsep + os.environ.get("PATH", "")


def java_major_version() -> int | None:
    try:
        result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=False)
        version_text = result.stderr or result.stdout
    except Exception:
        return None
    match = re.search(r'version "(\d+)', version_text)
    return int(match.group(1)) if match else None


def find_jdk17_home() -> Path | None:
    for root in [Path("C:/Program Files/Eclipse Adoptium"), Path("C:/Program Files/Java")]:
        if root.exists():
            for candidate in root.glob("**/jdk-17*"):
                if (candidate / "bin" / "java.exe").exists():
                    return candidate
    return None


current_java = java_major_version()
if current_java is not None and current_java > 17:
    jdk17_home = find_jdk17_home()
    if jdk17_home is None:
        raise RuntimeError("Instala JDK 17 para ejecutar PySpark correctamente.")
    os.environ["JAVA_HOME"] = str(jdk17_home)
    os.environ["PATH"] = str(jdk17_home / "bin") + os.pathsep + os.environ.get("PATH", "")

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Entrada normalizada: {NORMALIZED_BASE_DIR}")
print(f"Lookup zonas: {LOOKUP_DIR}")
print(f"Auditoria Gold: {AUDIT_LOG_PATH}")
print(f"HADOOP_HOME: {os.environ['HADOOP_HOME']}")

In [ ]:
from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pymongo import MongoClient, UpdateOne, ASCENDING
from pymongo.errors import ServerSelectionTimeoutError

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 4g "
    f"--conf spark.driver.extraJavaOptions=-Dhadoop.home.dir={HADOOP_HOME_JAVA} "
    f"--conf spark.executor.extraJavaOptions=-Dhadoop.home.dir={HADOOP_HOME_JAVA} "
    "pyspark-shell"
)

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("Gold_MongoDB_PowerBI_TLC")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.maxResultSize", "512m")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "128")
    .config("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))
    .config("spark.sql.parquet.columnarReaderBatchSize", "1024")
    .config("spark.driver.extraJavaOptions", f"-Dhadoop.home.dir={HADOOP_HOME_JAVA}")
    .config("spark.executor.extraJavaOptions", f"-Dhadoop.home.dir={HADOOP_HOME_JAVA}")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext._jvm.java.lang.System.setProperty("hadoop.home.dir", HADOOP_HOME_JAVA)
print("Spark master:", spark.sparkContext.master)
spark

## 2. Configuracion de MongoDB

In [ ]:
MONGODB_URI = os.getenv("MONGODB_URI", "mongodb://localhost:27017")
MONGODB_DATABASE = os.getenv("MONGODB_DATABASE", "tlc_trip_records_gold")
MONGO_BATCH_SIZE = 2000
REBUILD_GOLD_COLLECTIONS = True

COLLECTIONS = {
    "fact": "fact_viajes_agregados",
    "year": "dim_anio",
    "month": "dim_mes",
    "hour": "dim_hora",
    "taxi": "dim_tipo_taxi",
    "pickup_zone": "dim_zona_pickup",
    "dropoff_zone": "dim_zona_dropoff",
    "payment": "dim_pago",
    "ratecode": "dim_ratecode",
    "trip_type": "dim_trip_type",
    "shared_ride": "dim_shared_ride",
    "provider": "dim_proveedor",
    "audit": "audit_gold_powerbi",
}
LEGACY_COLLECTIONS = ["dim_tiempo"]

mongo_client = MongoClient(
    MONGODB_URI,
    serverSelectionTimeoutMS=10000,
    connectTimeoutMS=10000,
    socketTimeoutMS=120000,
)
try:
    mongo_client.admin.command("ping")
except ServerSelectionTimeoutError as error:
    raise ConnectionError(
        "No se pudo conectar a MongoDB. Inicia MongoDB local o configura MONGODB_URI para Atlas."
    ) from error

mongo_db = mongo_client[MONGODB_DATABASE]
print(f"Conexion MongoDB correcta. Base: {MONGODB_DATABASE}")
print(f"Reconstruir colecciones: {REBUILD_GOLD_COLLECTIONS}")

## 3. Catalogos desde diccionarios oficiales

In [ ]:
PAYMENT_DOCUMENTS = [
    {"_id": -1, "payment_type": -1, "forma_pago": "No informado / No aplica"},
    {"_id": 0, "payment_type": 0, "forma_pago": "Flex Fare trip"},
    {"_id": 1, "payment_type": 1, "forma_pago": "Credit card"},
    {"_id": 2, "payment_type": 2, "forma_pago": "Cash"},
    {"_id": 3, "payment_type": 3, "forma_pago": "No charge"},
    {"_id": 4, "payment_type": 4, "forma_pago": "Dispute"},
    {"_id": 5, "payment_type": 5, "forma_pago": "Unknown"},
    {"_id": 6, "payment_type": 6, "forma_pago": "Voided trip"},
]

RATECODE_DOCUMENTS = [
    {"_id": -1, "ratecode_id": -1, "ratecode": "No informado / No aplica"},
    {"_id": 1, "ratecode_id": 1, "ratecode": "Standard rate"},
    {"_id": 2, "ratecode_id": 2, "ratecode": "JFK"},
    {"_id": 3, "ratecode_id": 3, "ratecode": "Newark"},
    {"_id": 4, "ratecode_id": 4, "ratecode": "Nassau or Westchester"},
    {"_id": 5, "ratecode_id": 5, "ratecode": "Negotiated fare"},
    {"_id": 6, "ratecode_id": 6, "ratecode": "Group ride"},
    {"_id": 99, "ratecode_id": 99, "ratecode": "Null / unknown"},
]

TRIP_TYPE_DOCUMENTS = [
    {"_id": -1, "trip_type": -1, "tipo_viaje": "No informado / No aplica"},
    {"_id": 1, "trip_type": 1, "tipo_viaje": "Street-hail"},
    {"_id": 2, "trip_type": 2, "tipo_viaje": "Dispatch"},
]

SHARED_RIDE_DOCUMENTS = [
    {"_id": -1, "shared_ride_flag": -1, "viaje_compartido": "No aplica"},
    {"_id": 0, "shared_ride_flag": 0, "viaje_compartido": "No compartido / no informado"},
    {"_id": 1, "shared_ride_flag": 1, "viaje_compartido": "Compartido"},
]

TAXI_DOCUMENTS = [
    {"_id": "yellow", "tipo_dataset": "yellow", "tipo_taxi": "Yellow Taxi"},
    {"_id": "green", "tipo_dataset": "green", "tipo_taxi": "Green Taxi / SHL"},
    {"_id": "fhv", "tipo_dataset": "fhv", "tipo_taxi": "For-Hire Vehicle"},
]

VENDOR_LABELS = {
    "1": "Creative Mobile Technologies, LLC",
    "2": "Curb Mobility, LLC",
    "6": "Myle Technologies Inc",
    "7": "Helix",
}

## 4. Funciones auxiliares

In [ ]:
def append_audit(record: dict) -> None:
    payload = {
        "pipeline": "gold_mongodb_powerbi",
        "executed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        **record,
    }
    with AUDIT_LOG_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(payload, ensure_ascii=False, default=str) + "\n")


def clean_column_name(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_")


def quote_column(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"


def normalize_columns(df: DataFrame) -> DataFrame:
    used_names = set()
    aliases = []
    for original_name in df.columns:
        base_name = clean_column_name(original_name)
        new_name = base_name
        suffix = 2
        while new_name in used_names:
            new_name = f"{base_name}_{suffix}"
            suffix += 1
        used_names.add(new_name)
        aliases.append(F.col(quote_column(original_name)).alias(new_name))
    return df.select(*aliases)


def ensure_column(df: DataFrame, name: str, data_type: str) -> DataFrame:
    if name not in df.columns:
        return df.withColumn(name, F.lit(None).cast(data_type))
    return df


def numeric_or_zero(name: str):
    return F.coalesce(F.col(name).cast("double"), F.lit(0.0))


def avg_numeric(name: str):
    return F.avg(F.col(name).cast("double")).alias(f"promedio_{name}")


def parquet_part_files(path: Path):
    if path.is_file() and path.suffix.lower() == ".parquet":
        return [str(path)]
    files = sorted(file for file in path.glob("*.parquet") if file.is_file())
    if not files:
        files = sorted(file for file in path.rglob("*.parquet") if file.is_file())
    return [str(file) for file in files]


def dataframe_to_documents(df: DataFrame):
    return [row.asDict(recursive=True) for row in df.toLocalIterator()]


def bulk_upsert_documents(collection_name: str, documents: list[dict], batch_size: int = MONGO_BATCH_SIZE):
    if not documents:
        return 0
    collection = mongo_db[collection_name]
    written = 0
    for start in range(0, len(documents), batch_size):
        batch = documents[start:start + batch_size]
        operations = [UpdateOne({"_id": doc["_id"]}, {"$set": doc}, upsert=True) for doc in batch]
        if operations:
            result = collection.bulk_write(operations, ordered=False)
            written += result.upserted_count + result.modified_count + result.matched_count
    return written


def write_dataframe_to_mongo(df: DataFrame, collection_name: str):
    uri = MONGODB_URI
    db_name = MONGODB_DATABASE
    batch_size = MONGO_BATCH_SIZE

    def write_partition(rows):
        from pymongo import MongoClient, UpdateOne
        client = MongoClient(uri, serverSelectionTimeoutMS=10000, connectTimeoutMS=10000, socketTimeoutMS=120000)
        collection = client[db_name][collection_name]
        operations = []
        for row in rows:
            doc = row.asDict(recursive=True)
            operations.append(UpdateOne({"_id": doc["_id"]}, {"$set": doc}, upsert=True))
            if len(operations) >= batch_size:
                collection.bulk_write(operations, ordered=False)
                operations.clear()
        if operations:
            collection.bulk_write(operations, ordered=False)
        client.close()

    df.foreachPartition(write_partition)


def reset_gold_collections():
    if not REBUILD_GOLD_COLLECTIONS:
        return
    for collection_name in COLLECTIONS.values():
        mongo_db[collection_name].delete_many({})
    for collection_name in LEGACY_COLLECTIONS:
        mongo_db[collection_name].drop()
    print("Colecciones Gold limpiadas.")

## 5. Lectura de Silver normalizado

In [ ]:
normalized_datasets = sorted(
    path for path in NORMALIZED_BASE_DIR.iterdir()
    if path.is_dir() and path.name.lower().endswith(".parquet")
)
if not normalized_datasets:
    raise FileNotFoundError(f"No hay datasets normalizados en {NORMALIZED_BASE_DIR}")

print("Datasets normalizados:")
for path in normalized_datasets:
    print(f"- {path.name}")

normalized_paths = [str(path) for path in normalized_datasets]
raw_df = spark.read.option("mergeSchema", "true").parquet(*normalized_paths)

REQUIRED_COLUMNS = {
    "tipo_dataset": "string",
    "anio": "int",
    "pickup_month": "int",
    "pickup_hour": "int",
    "pickup_location_id": "int",
    "dropoff_location_id": "int",
    "payment_type": "int",
    "ratecode_id": "int",
    "trip_type": "int",
    "shared_ride_flag": "int",
    "vendor_id": "string",
    "dispatching_base_num": "string",
    "affiliated_base_number": "string",
    "distancia_millas": "double",
    "duracion_minutos": "double",
    "monto_base": "double",
    "monto_total": "double",
    "propina": "double",
    "peajes": "double",
    "extra": "double",
    "mta_tax": "double",
    "improvement_surcharge": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
    "cbd_congestion_fee": "double",
    "ehail_fee": "double",
}

prepared_df = raw_df
for column_name, data_type in REQUIRED_COLUMNS.items():
    prepared_df = ensure_column(prepared_df, column_name, data_type)

prepared_df = (
    prepared_df
    .withColumn("anio", F.coalesce(F.col("anio").cast("int"), F.lit(-1)))
    .withColumn("pickup_month", F.coalesce(F.col("pickup_month").cast("int"), F.lit(-1)))
    .withColumn("pickup_hour", F.coalesce(F.col("pickup_hour").cast("int"), F.lit(-1)))
    .withColumn("pickup_location_id", F.coalesce(F.col("pickup_location_id").cast("int"), F.lit(-1)))
    .withColumn("dropoff_location_id", F.coalesce(F.col("dropoff_location_id").cast("int"), F.lit(-1)))
    .withColumn("payment_type", F.coalesce(F.col("payment_type").cast("int"), F.lit(-1)))
    .withColumn("ratecode_id", F.coalesce(F.col("ratecode_id").cast("int"), F.lit(-1)))
    .withColumn("trip_type", F.coalesce(F.col("trip_type").cast("int"), F.lit(-1)))
    .withColumn(
        "shared_ride_flag",
        F.when(F.col("tipo_dataset") == "fhv", F.coalesce(F.col("shared_ride_flag").cast("int"), F.lit(0))).otherwise(F.lit(-1)),
    )
    .withColumn(
        "provider_id",
        F.coalesce(F.col("vendor_id").cast("string"), F.col("dispatching_base_num"), F.col("affiliated_base_number"), F.lit("NO_INFO")),
    )
    .withColumn("provider_key", F.concat_ws("|", F.col("tipo_dataset"), F.col("provider_id")))
)

print("Esquema preparado:")
prepared_df.printSchema()

## 6. Construccion y carga de dimensiones

In [ ]:
reset_gold_collections()

bulk_upsert_documents(COLLECTIONS["payment"], PAYMENT_DOCUMENTS)
bulk_upsert_documents(COLLECTIONS["ratecode"], RATECODE_DOCUMENTS)
bulk_upsert_documents(COLLECTIONS["trip_type"], TRIP_TYPE_DOCUMENTS)
bulk_upsert_documents(COLLECTIONS["shared_ride"], SHARED_RIDE_DOCUMENTS)
bulk_upsert_documents(COLLECTIONS["taxi"], TAXI_DOCUMENTS)

MONTH_NAMES = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre",
}

year_docs = [
    {"_id": int(row["anio"]), "anio": int(row["anio"]), "etiqueta_anio": str(int(row["anio"]))}
    for row in prepared_df.select("anio").where(F.col("anio") > 0).dropDuplicates(["anio"]).orderBy("anio").collect()
]
month_docs = [
    {
        "_id": month,
        "mes": month,
        "pickup_month": month,
        "nombre_mes": MONTH_NAMES[month],
        "mes_corto": MONTH_NAMES[month][:3],
        "orden_mes": month,
        "trimestre": ((month - 1) // 3) + 1,
    }
    for month in range(1, 13)
]
hour_docs = [
    {"_id": hour, "pickup_hour": hour, "hora": f"{hour:02d}:00", "franja": "Madrugada" if hour < 6 else "Manana" if hour < 12 else "Tarde" if hour < 18 else "Noche"}
    for hour in range(24)
] + [{"_id": -1, "pickup_hour": -1, "hora": "No informado", "franja": "No informado"}]

bulk_upsert_documents(COLLECTIONS["year"], year_docs)
bulk_upsert_documents(COLLECTIONS["month"], month_docs)
bulk_upsert_documents(COLLECTIONS["hour"], hour_docs)

lookup_files = sorted(LOOKUP_DIR.glob("*.parquet"))
if not lookup_files:
    raise FileNotFoundError(f"No se encontraron archivos Parquet en {LOOKUP_DIR}")

lookup_df = normalize_columns(spark.read.parquet(*[str(path) for path in lookup_files]))
zone_df = (
    lookup_df
    .select(
        F.col("locationid").cast("int").alias("location_id"),
        F.col("borough"),
        F.col("zone"),
        F.col("service_zone"),
    )
    .dropDuplicates(["location_id"])
)

pickup_zone_df = zone_df.select(
    F.col("location_id").alias("_id"),
    F.col("location_id").alias("pickup_location_id"),
    F.col("borough").alias("pickup_borough"),
    F.col("zone").alias("pickup_zone"),
    F.col("service_zone").alias("pickup_service_zone"),
)
dropoff_zone_df = zone_df.select(
    F.col("location_id").alias("_id"),
    F.col("location_id").alias("dropoff_location_id"),
    F.col("borough").alias("dropoff_borough"),
    F.col("zone").alias("dropoff_zone"),
    F.col("service_zone").alias("dropoff_service_zone"),
)

bulk_upsert_documents(COLLECTIONS["pickup_zone"], dataframe_to_documents(pickup_zone_df))
bulk_upsert_documents(COLLECTIONS["dropoff_zone"], dataframe_to_documents(dropoff_zone_df))

provider_df = (
    prepared_df
    .select("provider_key", "provider_id", "tipo_dataset")
    .dropDuplicates(["provider_key"])
    .withColumn("_id", F.col("provider_key"))
    .withColumn(
        "tipo_proveedor",
        F.when(F.col("tipo_dataset") == "fhv", F.lit("FHV base")).otherwise(F.lit("Taxi vendor")),
    )
    .withColumn(
        "proveedor",
        F.when(F.col("provider_id") == "1", F.lit(VENDOR_LABELS["1"]))
         .when(F.col("provider_id") == "2", F.lit(VENDOR_LABELS["2"]))
         .when(F.col("provider_id") == "6", F.lit(VENDOR_LABELS["6"]))
         .when(F.col("provider_id") == "7", F.lit(VENDOR_LABELS["7"]))
         .when(F.col("provider_id") == "NO_INFO", F.lit("No informado"))
         .otherwise(F.concat(F.lit("Base/Vendor "), F.col("provider_id"))),
    )
    .select("_id", "provider_key", "provider_id", "tipo_dataset", "tipo_proveedor", "proveedor")
)
write_dataframe_to_mongo(provider_df, COLLECTIONS["provider"])

print("Dimensiones cargadas en MongoDB.")

## 7. Tabla de hechos agregada

In [ ]:
GROUP_COLUMNS = [
    "anio",
    "pickup_month",
    "pickup_hour",
    "tipo_dataset",
    "pickup_location_id",
    "dropoff_location_id",
    "payment_type",
    "ratecode_id",
    "trip_type",
    "shared_ride_flag",
    "provider_key",
]

fact_df = (
    prepared_df
    .groupBy(*GROUP_COLUMNS)
    .agg(
        F.count(F.lit(1)).cast("long").alias("cantidad_viajes"),
        F.sum(numeric_or_zero("passenger_count")).alias("total_pasajeros"),
        F.sum(numeric_or_zero("distancia_millas")).alias("total_distancia_millas"),
        F.sum(numeric_or_zero("duracion_minutos")).alias("total_duracion_minutos"),
        F.sum(numeric_or_zero("monto_base")).alias("total_monto_base"),
        F.sum(numeric_or_zero("monto_total")).alias("total_monto_total"),
        F.sum(numeric_or_zero("propina")).alias("total_propina"),
        F.sum(numeric_or_zero("peajes")).alias("total_peajes"),
        F.sum(numeric_or_zero("extra")).alias("total_extra"),
        F.sum(numeric_or_zero("mta_tax")).alias("total_mta_tax"),
        F.sum(numeric_or_zero("improvement_surcharge")).alias("total_improvement_surcharge"),
        F.sum(numeric_or_zero("congestion_surcharge")).alias("total_congestion_surcharge"),
        F.sum(numeric_or_zero("airport_fee")).alias("total_airport_fee"),
        F.sum(numeric_or_zero("cbd_congestion_fee")).alias("total_cbd_congestion_fee"),
        F.sum(numeric_or_zero("ehail_fee")).alias("total_ehail_fee"),
        avg_numeric("distancia_millas"),
        avg_numeric("duracion_minutos"),
        avg_numeric("monto_total"),
        avg_numeric("propina"),
    )
    .withColumn(
        "_id",
        F.sha2(F.concat_ws("|", *[F.col(column).cast("string") for column in GROUP_COLUMNS]), 256),
    )
    .withColumn("loaded_at", F.current_timestamp())
    .select("_id", *GROUP_COLUMNS, *[column for column in [
        "cantidad_viajes",
        "total_pasajeros",
        "total_distancia_millas",
        "total_duracion_minutos",
        "total_monto_base",
        "total_monto_total",
        "total_propina",
        "total_peajes",
        "total_extra",
        "total_mta_tax",
        "total_improvement_surcharge",
        "total_congestion_surcharge",
        "total_airport_fee",
        "total_cbd_congestion_fee",
        "total_ehail_fee",
        "promedio_distancia_millas",
        "promedio_duracion_minutos",
        "promedio_monto_total",
        "promedio_propina",
        "loaded_at",
    ]])
)

fact_df = fact_df.persist(StorageLevel.DISK_ONLY)
fact_count = fact_df.count()
print(f"Filas agregadas para fact_viajes_agregados: {fact_count:,}")
fact_df.show(5, truncate=False)

## 8. Carga de hechos a MongoDB

In [ ]:
started_at = datetime.now(timezone.utc)
write_dataframe_to_mongo(fact_df, COLLECTIONS["fact"])
finished_at = datetime.now(timezone.utc)

audit_document = {
    "_id": f"gold_powerbi_{finished_at.strftime('%Y%m%d_%H%M%S')}",
    "pipeline": "gold_mongodb_powerbi",
    "started_at": started_at,
    "finished_at": finished_at,
    "database": MONGODB_DATABASE,
    "fact_collection": COLLECTIONS["fact"],
    "fact_rows": fact_count,
    "normalized_datasets": [path.name for path in normalized_datasets],
    "grain": GROUP_COLUMNS,
    "status": "completed",
}
bulk_upsert_documents(COLLECTIONS["audit"], [audit_document])
append_audit(audit_document)

print(f"Carga de hechos completada: {fact_count:,} documentos agregados")

## 9. Indices para MongoDB y Power BI

In [ ]:
fact_collection = mongo_db[COLLECTIONS["fact"]]
fact_collection.create_index([("anio", ASCENDING), ("pickup_month", ASCENDING)])
fact_collection.create_index([("pickup_hour", ASCENDING)])
fact_collection.create_index([("tipo_dataset", ASCENDING)])
fact_collection.create_index([("pickup_location_id", ASCENDING)])
fact_collection.create_index([("dropoff_location_id", ASCENDING)])
fact_collection.create_index([("payment_type", ASCENDING)])
fact_collection.create_index([("ratecode_id", ASCENDING)])
fact_collection.create_index([("provider_key", ASCENDING)])

mongo_db[COLLECTIONS["year"]].create_index([("anio", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["month"]].create_index([("pickup_month", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["hour"]].create_index([("pickup_hour", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["taxi"]].create_index([("tipo_dataset", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["pickup_zone"]].create_index([("pickup_location_id", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["dropoff_zone"]].create_index([("dropoff_location_id", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["payment"]].create_index([("payment_type", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["ratecode"]].create_index([("ratecode_id", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["trip_type"]].create_index([("trip_type", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["shared_ride"]].create_index([("shared_ride_flag", ASCENDING)], unique=True)
mongo_db[COLLECTIONS["provider"]].create_index([("provider_key", ASCENDING)], unique=True)

print(f"Base MongoDB: {MONGODB_DATABASE}")
for label, collection_name in COLLECTIONS.items():
    count = mongo_db[collection_name].count_documents({})
    print(f"- {collection_name}: {count:,} documentos")

sample = fact_collection.find_one({}, {"_id": 0})
print("\nEjemplo de hecho agregado:")
print(sample)

## 10. Relaciones para Power BI

Conecta Power BI a MongoDB usando MongoDB Atlas SQL, MongoDB BI Connector/ODBC o el conector disponible en tu entorno. En el modelo de Power BI crea relaciones **1 a muchos** desde cada dimension hacia `fact_viajes_agregados`:

| Dimension | Columna | Hechos | Columna |
|---|---|---|---|
| `dim_anio` | `anio` | `fact_viajes_agregados` | `anio` |
| `dim_mes` | `pickup_month` | `fact_viajes_agregados` | `pickup_month` |
| `dim_hora` | `pickup_hour` | `fact_viajes_agregados` | `pickup_hour` |
| `dim_tipo_taxi` | `tipo_dataset` | `fact_viajes_agregados` | `tipo_dataset` |
| `dim_zona_pickup` | `pickup_location_id` | `fact_viajes_agregados` | `pickup_location_id` |
| `dim_zona_dropoff` | `dropoff_location_id` | `fact_viajes_agregados` | `dropoff_location_id` |
| `dim_pago` | `payment_type` | `fact_viajes_agregados` | `payment_type` |
| `dim_ratecode` | `ratecode_id` | `fact_viajes_agregados` | `ratecode_id` |
| `dim_trip_type` | `trip_type` | `fact_viajes_agregados` | `trip_type` |
| `dim_shared_ride` | `shared_ride_flag` | `fact_viajes_agregados` | `shared_ride_flag` |
| `dim_proveedor` | `provider_key` | `fact_viajes_agregados` | `provider_key` |

Medidas sugeridas en Power BI:

```DAX
Viajes = SUM(fact_viajes_agregados[cantidad_viajes])
Ingresos = SUM(fact_viajes_agregados[total_monto_total])
Distancia = SUM(fact_viajes_agregados[total_distancia_millas])
Duracion = SUM(fact_viajes_agregados[total_duracion_minutos])
Ticket Promedio = DIVIDE([Ingresos], [Viajes])
Propina Promedio = DIVIDE(SUM(fact_viajes_agregados[total_propina]), [Viajes])
```


In [ ]:
fact_df.unpersist()
gc.collect()
spark.catalog.clearCache()
print("Recursos Spark liberados. Modelo Gold listo para Power BI.")